# F-10 — Extended feature engineering (D-67)

Stage 1: build 5 new feature families onto a new `forecast_daily_v3.csv` (additive clone of v2), verify each against a concrete checklist, then run a cheap leave-one-group-in signal check on a direct point forecast. Stage 2b: confirm (or overturn) Stage 1's finding against the real recursive-rollout ensemble (B-10), since that is the track this whole branch was meant to improve. See `F10_results.md` for the full write-up and `DECISIONS.md` D-67 (+ addendum)/D-68 for the empirical grounding and outcome.

Families: (a) livestock species disaggregation, (b) land-use regime flag (`fx_is_arable`), (c) catchment flow, (d) fertilizer/management richness, (e) bonus liveweight density. Scoped to forecasting only throughout -- the gap-filling pipeline (`gapfill_rfm.py`/F-08) is never touched.

## Step 1 — build bonus family (e): liveweight density

Runs `src/features/build_bodyweight_density.py`: last-observed-carried-forward per-animal location (resolved to real NWFP field codes) x weight join -> `data/Hourly/bodyweight_density.csv`.

In [1]:
import subprocess, sys
ROOT = r"c:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project"
r = subprocess.run([sys.executable, f"{ROOT}\\src\\features\\build_bodyweight_density.py"], cwd=ROOT, capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-2000:] if r.returncode else "")
assert r.returncode == 0

Cattle: 858 animals, 53,900 tower-resident animal-days, 53,900/53,900 (100.0%) have a resolvable weight
Breeding Sheep: 765 animals, 79,852 tower-resident animal-days, 76,302/79,852 (95.6%) have a resolvable weight
Lamb: 2341 animals, 81,399 tower-resident animal-days, 73,036/81,399 (89.7%) have a resolvable weight

Wrote C:\Users\Nicholas\Documents\COMP0191 MSc Artificial Intelligence for Sustainable Development Project\data\Hourly\bodyweight_density.csv (4,415 rows x 3 cols)
  Tower 2: 441 distinct days with a resolved liveweight-density value (1026.1 kg/ha mean when present)
  Tower 4: 1,730 distinct days with a resolved liveweight-density value (1114.5 kg/ha mean when present)
  Tower 9: 2,244 distinct days with a resolved liveweight-density value (964.8 kg/ha mean when present)




## Step 2 — build `forecast_daily_v3.csv`

Runs `src/features/build_forecasting_matrix_v3.py`: left-merges families (a)-(e) onto `forecast_daily_v2.csv` (read-only), writes `data/Hourly/forecast_daily_v3.csv`. Prints the row-count/column-identity assertions, all-NaN check, and per-tower `fx_is_arable` flip dates inline (script-internal verification).

In [2]:
r = subprocess.run([sys.executable, f"{ROOT}\\src\\features\\build_forecasting_matrix_v3.py"], cwd=ROOT, capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-2000:] if r.returncode else "")
assert r.returncode == 0

Loaded forecast_daily_v2.csv (8772, 48)
Loaded EXT (raw hourly, for species/flow columns) (70153, 524)
Verified: row count unchanged, every pre-existing v2 column byte-identical post-merge.
New columns added: ['fx_cattle_dens', 'fx_sheep_dens', 'fx_lamb_dens', 'fx_is_arable', 'fx_flow_mean', 'fx_flow_lag7', 'fx_flow_lag14', 'fx_flow_lag21', 'fx_flow_lag28', 'fx_flow_roll7', 'fx_flow_roll14', 'fx_mgmt_fertN_recency', 'fx_mgmt_fertN_rate', 'fx_mgmt_lime_recency', 'fx_mgmt_cultiv_recency', 'fx_mgmt_cut_recency', 'fx_mgmt_manure_recency', 'fx_total_liveweight_dens']
All-NaN new columns: none

Land-use flip dates (fx_is_arable):
  Tower 2: 2019-09-09 00:00:00
  Tower 4: never flips (fx_is_arable=0 throughout)
  Tower 9: never flips (fx_is_arable=0 throughout)

Per-tower non-null coverage of new fx_ columns (%):
  Tower 2:
fx_cattle_dens              100.0
fx_sheep_dens               100.0
fx_lamb_dens                100.0
fx_is_arable                100.0
fx_flow_mean                 92.0
f

## Step 3 — verification checklist (per F-10 plan)

Independent, notebook-level checks beyond what the builder script already asserts internally.

In [3]:
import pandas as pd, numpy as np
v3 = pd.read_csv(f"{ROOT}\\data\\Hourly\\forecast_daily_v3.csv", low_memory=False)

# (a) LSU-linearity exact identity
lhs = v3["fx_lsu_dens"]
rhs = 1.0*v3["fx_cattle_dens"] + 0.1*v3["fx_sheep_dens"] + 0.05*v3["fx_lamb_dens"]
maxdiff = (lhs-rhs).abs().max()
print("(a) max |fx_lsu_dens - weighted species sum| =", maxdiff)
assert maxdiff < 1e-9

# (b) is_arable: T4/T9 always 0, T2 flips 2019-09-09
assert v3[v3.tower==4].fx_is_arable.sum() == 0
assert v3[v3.tower==9].fx_is_arable.sum() == 0
t2 = v3[v3.tower==2].sort_values("Datetime")
flip_rows = t2[(t2.Datetime>="2019-09-08") & (t2.Datetime<="2019-09-10")][["Datetime","fx_is_arable"]]
print("(b) T4/T9 fx_is_arable always 0: OK")
print(flip_rows)

# (c) flow lag/roll structural check
sub = v3[v3.tower==2].sort_values("Datetime").reset_index(drop=True)
manual_roll7 = sub["fx_flow_mean"].rolling(7, min_periods=1).mean()
print("(c) roll7 matches manual recompute:", np.allclose(sub["fx_flow_roll7"].dropna(), manual_roll7.dropna()))
for t in [2,4,9]:
    cov = v3[v3.tower==t]["fx_flow_mean"].notna().mean()*100
    print(f"(c) Tower {t} fx_flow_mean coverage: {cov:.1f}%")
    assert cov >= 85

print("\nAll checklist assertions passed.")

(a) max |fx_lsu_dens - weighted species sum| = 1.3322676295501878e-15
(b) T4/T9 fx_is_arable always 0: OK
       Datetime  fx_is_arable
980  2019-09-08           0.0
981  2019-09-09           1.0
982  2019-09-10           1.0
(c) roll7 matches manual recompute: True
(c) Tower 2 fx_flow_mean coverage: 92.0%
(c) Tower 4 fx_flow_mean coverage: 88.5%
(c) Tower 9 fx_flow_mean coverage: 89.1%

All checklist assertions passed.


## Step 4 — Stage 1 signal check

Runs `f10_signal_check.py`: cheap, bounded, single-seed leave-one-group-in RF ablation, all 3 towers, h in {1, 14}, plus a follow-up swap test for the species family. See `F10_results.md` for the full table and the go/no-go verdict.

In [4]:
r = subprocess.run([sys.executable, f"{ROOT}\\notebooks\\04_feature_engineering\\f10_signal_check.py"], cwd=ROOT, capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-2000:] if r.returncode else "")
assert r.returncode == 0

BASE_FX (34): ['fx_WS_mean', 'fx_USTAR_mean', 'fx_TA_mean', 'fx_TA_min', 'fx_TA_max', 'fx_VPD_mean', 'fx_SWIN_mean', 'fx_RN_mean', 'fx_PPFD_mean', 'fx_SWC_mean', 'fx_TS_mean', 'fx_SHF_mean', 'fx_PRECIP_sum', 'fx_wd_sin', 'fx_wd_cos', 'fx_SWC_lag7', 'fx_TS_lag7', 'fx_SWC_lag14', 'fx_TS_lag14', 'fx_SWC_lag21', 'fx_TS_lag21', 'fx_SWC_lag28', 'fx_TS_lag28', 'fx_SWC_roll7', 'fx_TS_roll7', 'fx_SWC_roll14', 'fx_TS_roll14', 'fx_DOY_sin', 'fx_DOY_cos', 'fx_is_growing', 'fx_is_winter', 'fx_lsu_dens', 'fx_grazing_active', 'fx_days_since_grazing']
New columns by family: {'species': ['fx_cattle_dens', 'fx_sheep_dens', 'fx_lamb_dens'], 'arable': ['fx_is_arable'], 'flow': ['fx_flow_mean', 'fx_flow_lag7', 'fx_flow_lag14', 'fx_flow_lag21', 'fx_flow_lag28', 'fx_flow_roll7', 'fx_flow_roll14'], 'mgmt': ['fx_mgmt_fertN_recency', 'fx_mgmt_fertN_rate', 'fx_mgmt_lime_recency', 'fx_mgmt_cultiv_recency', 'fx_mgmt_cut_recency', 'fx_mgmt_manure_recency'], 'bodyweight': ['fx_total_liveweight_dens']}

=== h=1: trai

## Step 5 — Stage 2b: recursive-rollout confirmation (run despite Stage 1's null result)

Stage 1 only exercised a direct, non-autoregressive point forecast. Per direct user instruction ("run the rollout test (recursive forecasting). The point of this experiment is to test on forecasting performance improvements, no gap-filing"), the real B-10 recursive-rollout ensemble (RF+XGB+LightGBM+SARIMAX) was also tested — all 3 towers, all 5 anchors, 6 configs (`BASE` + each of the 5 families) — via `notebooks/05_benchmarking/b16_recursive_rollout_v3.py` (not re-run in this notebook; ~28 minutes, see `DECISIONS.md` D-67 addendum and `F10_results.md` for the full table).

**Result: agrees with Stage 1.** `BASE`'s `Ensemble_unweighted` reproduces `BEST_RESULTS.md`'s published headline almost exactly (R²=-0.1652 vs. -0.165). None of the 5 families beat `BASE` on the ensemble or on any individual tree model. The species family's tower-specific pattern (helps T4, hurts T9) replicated independently in this second, methodologically distinct evaluation.

## Verdict

None of the 5 families clear Stage 1's pre-registered go/no-go bar (consistent Delta R2 > +0.01 across towers/horizons, or a top-10 SHAP rank that survives a collinearity check). Two real implementation bugs were caught and fixed during this verification pass (a false-positive `fx_is_arable` trigger from an over-broad management-event channel; an all-NaN `fx_flow_lag*` indexing bug). Stage 2b (recursive-rollout confirmation, Step 5 above) was run anyway per direct user instruction and **agrees**: no family improves the real B-10 ensemble either. **Both forecasting tracks (point-forecast and recursive-rollout) confirm no feature family is worth adopting -- `BEST_RESULTS.md` is unchanged.** The gap-filling pipeline (`gapfill_rfm.py`/F-08) was never touched by any part of F-10; this branch was scoped to forecasting performance only, per direct user instruction. See `F10_results.md` for the full tables and honest discussion of caveats, and `DECISIONS.md` D-67 (+ addendum)/D-68 for the logged decisions.